# Bước 4: Tìm đường dẫn Ảnh đúng

Tìm đường dẫn cho tất cả 69,863 images trong metadata.

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict

In [ ]:
# Load metadata
METADATA_PATH = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Total metadata records: {len(metadata)}")

# Lấy danh sách filenames
all_filenames = set(m['name'] for m in metadata)
print(f"Unique filenames: {len(all_filenames)}")

In [ ]:
# Tìm tất cả ảnh trong dataset
DATASET_BASE = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"

print("Building image index...")
image_map = {}  # filename → full_path

count = 0
for root, dirs, files in os.walk(DATASET_BASE):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_map[f] = os.path.join(root, f)
            count += 1

print(f"Total images found: {count}")
print(f"Images in map: {len(image_map)}")

In [ ]:
# Kiểm tra metadata filenames có trong image_map không
matched = 0
unmatched = []

for meta in metadata[:100]:  # Check 100 đầu
    filename = meta['name']
    if filename in image_map:
        matched += 1
    else:
        unmatched.append(filename)

print(f"Matched: {matched}/100")

if unmatched:
    print(f"\nUnmatched samples (first 10):")
    for f in unmatched[:10]:
        print(f"  {f}")

In [ ]:
# Check tất cả metadata
matched_all = 0
for meta in metadata:
    if meta['name'] in image_map:
        matched_all += 1

print(f"Total matched: {matched_all}/{len(metadata)} ({matched_all/len(metadata)*100:.1f}%)")

In [ ]:
# Tạo danh sách valid samples (có ảnh + có detection labels)
valid_samples = []
no_image = 0
no_labels = 0
no_detection = 0

for meta in metadata:
    filename = meta['name']
    labels = meta.get('labels', [])

    # Check ảnh tồn tại
    if filename not in image_map:
        no_image += 1
        continue

    # Check có detection labels
    has_detection = any('box2d' in label for label in labels)
    if not has_detection:
        no_detection += 1
        continue

    # Valid sample
    valid_meta = meta.copy()
    valid_meta['image_path'] = image_map[filename]
    valid_samples.append(valid_meta)

print(f"Valid samples: {len(valid_samples)}/{len(metadata)}")
print(f"  - No image: {no_image}")
print(f"  - No detection labels: {no_detection}")

In [ ]:
# Tổng kết
print("=" * 60)
print("KẾT QUẢ:")
print("=" * 60)
print(f"Total metadata records: {len(metadata)}")
print(f"Images found on disk: {len(image_map)}")
print(f"Valid samples (image + detection): {len(valid_samples)}")
print(f"Valid rate: {len(valid_samples)/len(metadata)*100:.1f}%")

print("\n" + "=" * 60)
print("ĐƯỜNG DẪN ĐỂ SỬ DỤNG:")
print("=" * 60)

# In ra đường dẫn của một vài valid samples
print("\nSample valid paths:")
for i, sample in enumerate(valid_samples[:5]):
    print(f"  {i+1}. {sample['image_path']}")

## Kết luận

Nếu valid_samples > 0, có thể bắt đầu experiment!
Cần update code để sử dụng `image_path` từ valid_samples.